In [5]:
user_item_matrix = {
    'Peter': {'A': 5, 'B': 3, 'C': 4, 'D': 4, 'E': 3, 'F': 3, 'G': 4, 'H': 2, 'I': 1, 'J': 4},
    'John': {'A': 3, 'B': 4, 'C': 4, 'D': 5, 'E': 4, 'F': 5, 'G': 2, 'H': 2, 'I': 4, 'J': 1},
    'Kavita': {'A': 4, 'B': 5, 'C': 1, 'D': 2, 'E': 4, 'F': 3, 'G': 4, 'H': 2, 'I': 3, 'J': 4},
    'Mary': {'A': 4, 'B': 3, 'C': 2, 'D': 3, 'E': 5, 'F': 4, 'G': 3, 'H': 4, 'I': 2, 'J': 4},
    'Shyam': {'A': 3, 'B': 2, 'C': 3, 'D': 2, 'E': 1, 'F': 3, 'G': 3, 'H': 3, 'I': 4, 'J': 2},
}

In [6]:
import pandas as pd

df = pd.DataFrame(user_item_matrix).T
df

,A,B,C,D,E,F,G,H,I,J
Peter,5,3,4,4,3,3,4,2,1,4
John,3,4,4,5,4,5,2,2,4,1
Kavita,4,5,1,2,4,3,4,2,3,4
Mary,4,3,2,3,5,4,3,4,2,4
Shyam,3,2,3,2,1,3,3,3,4,2


In [41]:
pipeline_results = {
    'John': "A D E G".split(),
}
pipeline_results

{'John': ['A', 'D', 'E', 'G']}

### Using Simple Collaborative Filtering (only aggregating similar user ratings)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_top_k_neighbors(user_item_matrix, target_user, k=3):
    users = list(user_item_matrix.keys())
    target_vector = np.array([user_item_matrix[target_user][item] for item in sorted(user_item_matrix[target_user].keys())]).reshape(1, -1)
    
    similarities = {}
    for user in users:
        if user != target_user:
            user_vector = np.array([user_item_matrix[user][item] for item in sorted(user_item_matrix[user].keys())]).reshape(1, -1)
            sim = cosine_similarity(target_vector, user_vector)[0][0]
            similarities[user] = sim
    
    top_k_neighbors = sorted(similarities, key=similarities.get, reverse=True)[:k]
    return top_k_neighbors


def collaborative_filtering_recommendation(user_item_matrix, target_user, k=3):
    top_k_neighbors = get_top_k_neighbors(user_item_matrix, target_user, k)
    
    # Aggregate ratings from neighbors
    recommendation_scores = {}
    for neighbor in top_k_neighbors:
        for item, rating in user_item_matrix[neighbor].items():
            if item not in recommendation_scores:
                recommendation_scores[item] = 0
            recommendation_scores[item] += rating
    
    # Sort items based on aggregated scores
    recommended_items = sorted(recommendation_scores, key=recommendation_scores.get, reverse=True)
    
    return recommended_items



In [12]:
get_top_k_neighbors(user_item_matrix, 'John', k=3)

['Shyam', 'Mary', 'Peter']

In [17]:
cf_results = collaborative_filtering_recommendation(user_item_matrix, 'John')
cf_results

['A', 'F', 'G', 'J', 'C', 'D', 'E', 'H', 'B', 'I']

In [19]:
# filter pipeline results of the target user with respect to the cf results
modified_results = [item for item in cf_results if item in pipeline_results['John']]
modified_results

['A', 'G', 'D', 'E']

### Using Collaborative Filtering with Pearson's Coefficient

In [29]:
def pearson_correlation(user_ratings, other_user_ratings):
    common_items = set(user_ratings.keys()) & set(other_user_ratings.keys())
    if not common_items:
        print("No common items rated by both users, returning 0 for similarity.")
        return 0

    n = len(common_items)
    mean_x = sum(user_ratings[item] for item in common_items) / n
    mean_y = sum(other_user_ratings[item] for item in common_items) / n

    numerator = sum(
        [
            (user_ratings[item] - mean_x) * (other_user_ratings[item] - mean_y)
            for item in common_items
        ]
    )

    denominator = (
        sum((user_ratings[item] - mean_x) ** 2 for item in common_items) ** 0.5
        * sum((other_user_ratings[item] - mean_y) ** 2 for item in common_items) ** 0.5
    )

    if denominator == 0:
        print("Denominator is zero, returning 0 for similarity.")
        return 0

    return numerator / denominator


In [ ]:
def predict_rating(user_ratings, sorted_users):
    recommendations = {}

    all_items = set(
        item for item_ratings in user_item_matrix.values() for item in item_ratings.keys()
    )

    user_items = set(user_ratings.keys())

    n = len(user_items)
    mean_x = sum(user_ratings[item] for item in user_items) / n
    sim_sum = sum(abs(sim) for _, sim in sorted_users if sim > 0)

    for similar_user, sim in sorted_users:
        if similar_user not in user_item_matrix:
            continue

        item_ratings = user_item_matrix[similar_user]
        mean_y = (
            sum(item_ratings[item] for item in all_items if item in item_ratings) / n
        )

        for item, item_rating in item_ratings.items():
            if item_rating > 0:
                if item not in recommendations:
                    recommendations[item] = mean_x

                recommendations[item] += sim * (item_rating - mean_y)

    # Normalize recommendations
    for item in recommendations:
        recommendations[item] /= sim_sum if sim_sum > 0 else 1

    return recommendations

In [31]:

def get_user_collaborative_filter(user_id, rating_matrix):
    if user_id not in rating_matrix:
        return []

    user_ratings = rating_matrix[user_id]
    similar_users = {}

    for other_user, ratings in rating_matrix.items():
        if other_user == user_id:
            continue
        similarity = pearson_correlation(user_ratings, ratings)
        similar_users[other_user] = similarity

    sorted_users = sorted(similar_users.items(), key=lambda x: x[1], reverse=True)[:3]  # TOP 3
    print(f"Similar users for {user_id}: {sorted_users}")

    recommendations = predict_rating(user_ratings, sorted_users)

    recommended_items = sorted(
        recommendations.items(), key=lambda x: x[1], reverse=True
    )

    return recommended_items

In [39]:
recommendations = get_user_collaborative_filter('John', user_item_matrix)
recommendations

Similar users for John: [('Shyam', -0.039043440472151504), ('Peter', -0.15617376188860607), ('Mary', -0.221519407392074)]


[('I', 4.014666006031685),
 ('C', 3.5851881608380185),
 ('B', 3.558885955806702),
 ('H', 3.454496869831083),
 ('D', 3.402712193918096),
 ('G', 3.3636687534459444),
 ('F', 3.2983231079424766),
 ('J', 3.1811927865260223),
 ('E', 3.1548905814947057),
 ('A', 2.9859755841652644)]

In [40]:
cf_results = [tup[0] for tup in recommendations]
cf_results

['I', 'C', 'B', 'H', 'D', 'G', 'F', 'J', 'E', 'A']

In [42]:
# filter pipeline results of the target user with respect to the cf results
modified_results = [item for item in cf_results if item in pipeline_results['John']]
modified_results

['D', 'G', 'E', 'A']